# Object Separator — Colab training (Option C)

Trains Sarowar's class-agnostic object separator (3.94M params) on a
free Colab **T4 GPU**: 512x512 input, 40,000-image COCO subset,
24 epochs. Runtime -> Change runtime type -> **T4 GPU** first!

**If the session dies:** just re-run the cells from the top.
Checkpoints live on Google Drive and `--resume` continues training
where it stopped — nothing restarts from zero.

In [ ]:
# 1) Mount Google Drive. Checkpoints (last.pt / best.pt) are written
#    to a folder on Drive, so they survive a dead Colab session.
from google.colab import drive
drive.mount('/content/drive')

import os
CKPT_DIR = '/content/drive/MyDrive/object-separator-ckpt'
os.makedirs(CKPT_DIR, exist_ok=True)
print('Checkpoints ->', CKPT_DIR)

In [ ]:
# 2) Get the code and install the few packages Colab does not ship.
#    torch is preinstalled on Colab -- do NOT reinstall it.
#    (No pycocotools needed: the labels are plain JSON polygons.)
import os
REPO = '/content/object-separator'
if os.path.exists(REPO):
    !cd {REPO} && git pull
else:
    !git clone https://github.com/bluu6964/object-separator.git {REPO}
!pip install -q onnx onnxruntime
print('Repo at', REPO)

In [ ]:
# 3) Download the Option C data pack from the GitHub Release and
#    unpack it to /content/data. Pack layout (exactly what
#    train.py --data expects):
#      data/images/{train,val}/*.jpg
#      data/labels/{train,val}/*.json  ({"instances":[{"bbox","poly"}]})
#      data/cache-512/{train,val}/*.npz  (instance masks rasterized
#          ONCE by build_cache.py -- without them, every epoch
#          re-draws every polygon and the GPU starves)
#      data/manifest.json
#    Every tar part is verified against pack-manifest.json (exact
#    byte size + md5) and re-downloaded if it arrives truncated --
#    a truncated part used to silently break the tar unpack.
import hashlib, json, os, urllib.request

DATA_DIR = '/content/data'
API = ('https://api.github.com/repos/bluu6964/object-separator'
       '/releases/tags/data-coco-seg-40k')

def md5_of(path):
    h = hashlib.md5()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 22), b''):
            h.update(chunk)
    return h.hexdigest()

if (os.path.exists(os.path.join(DATA_DIR, 'manifest.json'))
        and os.path.exists(os.path.join(DATA_DIR, 'cache-512'))):
    print('Data + mask cache already present, skipping download.')
else:
    with urllib.request.urlopen(API) as r:
        rel = json.load(r)
    assets = {a['name']: a['browser_download_url'] for a in rel['assets']}
    os.makedirs('/content/pack', exist_ok=True)
    pm_path = '/content/pack/pack-manifest.json'
    urllib.request.urlretrieve(assets['pack-manifest.json'], pm_path)
    with open(pm_path) as f:
        pack = json.load(f)
    for part in pack['parts']:
        dest = '/content/pack/' + part['name']
        for attempt in range(3):
            if (os.path.exists(dest)
                    and os.path.getsize(dest) == part['bytes']
                    and md5_of(dest) == part['md5']):
                break
            print(f"Downloading {part['name']} "
                  f"({part['bytes'] / 1e6:.0f} MB), attempt {attempt + 1}")
            if os.path.exists(dest):
                os.remove(dest)
            urllib.request.urlretrieve(assets[part['name']], dest)
        assert (os.path.getsize(dest) == part['bytes']
                and md5_of(dest) == part['md5']), \
            f"{part['name']} failed size/md5 verification 3 times"
        print('Verified', part['name'])
    !cat /content/pack/data.tar.part-* | tar -xf - -C /content
    !rm -rf /content/pack
with open(os.path.join(DATA_DIR, 'manifest.json')) as f:
    man = json.load(f)
print('Staged:', man['staged'])
print('Mask cache entries (train):',
      len(os.listdir(os.path.join(DATA_DIR, 'cache-512', 'train'))))


In [ ]:
# 4) GPU check + a 30-step throughput probe (writes no checkpoints).
#    The img/s it prints is the real speed of THIS session -- the
#    full run is 40,000 images x 24 epochs at that rate.
import torch
print('GPU:', torch.cuda.get_device_name(0)
      if torch.cuda.is_available()
      else 'NONE -- switch the runtime to T4 GPU first!')
!cd /content/object-separator && python -u train.py --data /content/data --img 512 --batch 32 --epochs 24 --workers 2 --max-steps 30 --out /content/probe-runs

In [ ]:
# 5) TRAIN -- Option C: 512px, batch 32, 24 epochs TOTAL.
#    --resume + the Drive checkpoint folder: if the session dies,
#    re-run the cells from the top and training continues where it
#    stopped. last.pt is also refreshed every ~25 minutes
#    mid-epoch, so at most ~25 minutes of work is ever lost.
!cd /content/object-separator && python -u train.py --data /content/data --img 512 --batch 32 --epochs 24 --workers 2 --eval-limit 500 --save-every-min 25 --resume --ckpt-dir {CKPT_DIR}

In [ ]:
# 6) FINISH -- run this after training completes (or any time
#    best.pt exists on Drive): export ONNX, build layer-split
#    demos on 5 validation images, and print the summary.
#    Training already wrote last.pt / best.pt / metrics.json into
#    CKPT_DIR; this cell adds best.onnx and samples/ next to them.
import glob, json, os, shutil

# Export ONNX from the best checkpoint.
!cd /content/object-separator && python export.py --ckpt {CKPT_DIR}/best.pt --out {CKPT_DIR} --img 512

# Layer-split demos (original / colour viz / RGBA cutouts per object).
demo_src = '/content/demo_src'
os.makedirs(demo_src, exist_ok=True)
for p in sorted(glob.glob('/content/data/images/val/*.jpg'))[:5]:
    shutil.copy(p, demo_src)
!cd /content/object-separator && python predict.py --model {CKPT_DIR}/best.onnx --src /content/demo_src --out {CKPT_DIR}/samples --img 512 --conf 0.25

# Summary.
with open(os.path.join(CKPT_DIR, 'metrics.json')) as f:
    m = json.load(f)
onnx_path = os.path.join(CKPT_DIR, 'best.onnx')
print('Best val mask mAP@0.5 :', m.get('best_val_mask_map50'))
print('Final val mask mAP@0.5:', m.get('final_val_mask_map50'))
print('ONNX:', onnx_path, f"{os.path.getsize(onnx_path) / 1e6:.1f} MB")
print('Samples in', os.path.join(CKPT_DIR, 'samples'))